# Project 3 Data Processing: Observed 2025 Baseline + CMIP6 2026–2036 Projections

**Goal:** prepare data for an interactive geospatial visualization:

- Start with **NOAA nClimDiv county observed average temperature in 2025**
- Add **CMIP6 GFDL-ESM4 near-surface air temperature (`tas`) projections for 2026–2036**
- Calculate county/state warming relative to the observed 2025 baseline
- Export CSV + GeoJSON files for D3:
  - U.S. state overview map
  - Click state → zoom into counties
  - Hover/click county → show details and scenario line chart

**Research question draft:**  
> Compared with observed temperatures in 2025, how much are U.S. counties projected to warm by 2036 under different emissions scenarios?

## 0. Colab setup

Run this cell first in Google Colab. It installs geospatial and CMIP6 access packages.

In [ ]:
%%capture
!pip install geopandas pyogrio shapely rtree fsspec gcsfs zarr xarray dask netCDF4 cftime intake-esm cartopy

## 1. Imports and configuration

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import intake

warnings.filterwarnings("ignore")

OUTPUT_DIR = Path("/content/project3_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# NOAA nClimDiv county average temperature file.
# This is a public direct-download text file.
NOAA_TMPCCY_URL = "https://www.ncei.noaa.gov/pub/data/cirs/climdiv/climdiv-tmpccy-v1.0.0-20260506"

# Census cartographic boundary files.
# 20m is smaller and better for web maps; use 5m/500k if you need more detail.
COUNTY_ZIP_URL = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_county_20m.zip"
STATE_ZIP_URL = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_state_20m.zip"

PROJECTED_YEARS = list(range(2026, 2037))  # 2026–2036 inclusive
SCENARIOS = ["ssp126", "ssp245", "ssp585"]
SCENARIO_LABELS = {
    "ssp126": "Low emissions (SSP126)",
    "ssp245": "Medium emissions (SSP245)",
    "ssp585": "High emissions (SSP585)",
}

# Keep 50 states only. Exclude DC and territories for this project.
STATE_ABBRS_50 = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS",
    "KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY",
    "NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV",
    "WI","WY"
}

## 2. Load U.S. state and county boundaries

These GeoJSON files will be used later in D3. The county file provides the polygons for the zoomed-in view.

In [ ]:
states = gpd.read_file(STATE_ZIP_URL).to_crs("EPSG:4326")
counties = gpd.read_file(COUNTY_ZIP_URL).to_crs("EPSG:4326")

# Keep only 50 states.
states = states[states["STUSPS"].isin(STATE_ABBRS_50)].copy()

# Join state names/abbreviations onto counties.
state_lookup = states[["STATEFP", "NAME", "STUSPS"]].rename(
    columns={"NAME": "state", "STUSPS": "state_abbr"}
)

counties = counties.merge(state_lookup, on="STATEFP", how="inner")
counties = counties[counties["state_abbr"].isin(STATE_ABBRS_50)].copy()

# Rename county name column for clarity.
counties = counties.rename(columns={"NAME": "county"})

# Make sure key columns are strings.
counties["GEOID"] = counties["GEOID"].astype(str).str.zfill(5)
counties["COUNTYFP"] = counties["COUNTYFP"].astype(str).str.zfill(3)
counties["STATEFP"] = counties["STATEFP"].astype(str).str.zfill(2)

print("States:", len(states))
print("Counties:", len(counties))
counties[["GEOID", "STATEFP", "COUNTYFP", "county", "state", "state_abbr"]].head()

## 3. Load NOAA nClimDiv 2025 county observed average temperature

The NOAA file is fixed-width text. Each row contains:
- NOAA state code
- county code within state
- element code
- year
- 12 monthly values

For `climdiv-tmpccy`, the element is average temperature. We calculate the annual 2025 county average by averaging Jan–Dec.

In [ ]:
# Fixed-width column positions from NOAA county readme.
colspecs = [
    (0, 2),    # NOAA state code
    (2, 5),    # county code within state
    (5, 7),    # element code; 02 = average temperature
    (7, 11),   # year
    (11, 18),  # Jan
    (18, 25),  # Feb
    (25, 32),  # Mar
    (32, 39),  # Apr
    (39, 46),  # May
    (46, 53),  # Jun
    (53, 60),  # Jul
    (60, 67),  # Aug
    (67, 74),  # Sep
    (74, 81),  # Oct
    (81, 88),  # Nov
    (88, 95),  # Dec
]

names = [
    "noaa_state_code", "county_fp", "element_code", "year",
    "jan", "feb", "mar", "apr", "may", "jun",
    "jul", "aug", "sep", "oct", "nov", "dec"
]

obs_raw = pd.read_fwf(
    NOAA_TMPCCY_URL,
    colspecs=colspecs,
    names=names,
    dtype={
        "noaa_state_code": str,
        "county_fp": str,
        "element_code": str,
        "year": int,
    }
)

obs_raw["noaa_state_code"] = obs_raw["noaa_state_code"].str.zfill(2)
obs_raw["county_fp"] = obs_raw["county_fp"].str.zfill(3)
obs_raw["element_code"] = obs_raw["element_code"].str.zfill(2)

month_cols = [
    "jan", "feb", "mar", "apr", "may", "jun",
    "jul", "aug", "sep", "oct", "nov", "dec"
]

# Keep 2025 average temperature rows.
obs_2025 = obs_raw[
    (obs_raw["year"] == 2025) &
    (obs_raw["element_code"] == "02")
].copy()

# Missing monthly values are -99.99 for temperature.
obs_2025[month_cols] = obs_2025[month_cols].replace(-99.99, np.nan)

obs_2025["observed_temp_2025_f"] = obs_2025[month_cols].mean(axis=1)
obs_2025["observed_temp_2025_c"] = (
    obs_2025["observed_temp_2025_f"] - 32
) * 5 / 9

print("NOAA 2025 county rows:", len(obs_2025))
obs_2025.head()

### NOAA state code mapping

Important: NOAA state codes in this file are not the same as Census state FIPS codes.  
We map NOAA state code → state name, then merge with Census county polygons using:

`state name + county_fp`

In [ ]:
noaa_state_code_to_name = {
    "01": "Alabama",
    "02": "Arizona",
    "03": "Arkansas",
    "04": "California",
    "05": "Colorado",
    "06": "Connecticut",
    "07": "Delaware",
    "08": "Florida",
    "09": "Georgia",
    "10": "Idaho",
    "11": "Illinois",
    "12": "Indiana",
    "13": "Iowa",
    "14": "Kansas",
    "15": "Kentucky",
    "16": "Louisiana",
    "17": "Maine",
    "18": "Maryland",
    "19": "Massachusetts",
    "20": "Michigan",
    "21": "Minnesota",
    "22": "Mississippi",
    "23": "Missouri",
    "24": "Montana",
    "25": "Nebraska",
    "26": "Nevada",
    "27": "New Hampshire",
    "28": "New Jersey",
    "29": "New Mexico",
    "30": "New York",
    "31": "North Carolina",
    "32": "North Dakota",
    "33": "Ohio",
    "34": "Oklahoma",
    "35": "Oregon",
    "36": "Pennsylvania",
    "37": "Rhode Island",
    "38": "South Carolina",
    "39": "South Dakota",
    "40": "Tennessee",
    "41": "Texas",
    "42": "Utah",
    "43": "Vermont",
    "44": "Virginia",
    "45": "Washington",
    "46": "West Virginia",
    "47": "Wisconsin",
    "48": "Wyoming",
    "49": "Hawaii",
    "50": "Alaska",
}

obs_2025["state"] = obs_2025["noaa_state_code"].map(noaa_state_code_to_name)

obs_2025_clean = obs_2025[
    [
        "state",
        "county_fp",
        "observed_temp_2025_f",
        "observed_temp_2025_c",
    ]
].copy()

obs_2025_clean.head()

## 4. Merge observed 2025 temperature onto county polygons

This creates the county metadata table used for CMIP6 sampling and final D3 data.

In [ ]:
county_meta = counties[
    [
        "GEOID", "STATEFP", "COUNTYFP", "county", "state", "state_abbr", "ALAND", "geometry"
    ]
].copy()

county_meta = county_meta.merge(
    obs_2025_clean,
    left_on=["state", "COUNTYFP"],
    right_on=["state", "county_fp"],
    how="left"
)

missing_obs = county_meta[county_meta["observed_temp_2025_c"].isna()]
print("Counties missing observed 2025 temp:", len(missing_obs))

if len(missing_obs) > 0:
    display(missing_obs[["GEOID", "county", "state", "state_abbr", "COUNTYFP"]].head(20))

# For safety, drop counties that cannot be matched.
# Ideally this should be 0 or very small.
county_meta = county_meta.dropna(subset=["observed_temp_2025_c"]).copy()

# Representative point is safer than centroid because it stays inside the polygon.
# This is used to assign each county to the nearest CMIP6 model grid cell.
rep_points = county_meta.geometry.representative_point()
county_meta["rep_lon"] = rep_points.x
county_meta["rep_lat"] = rep_points.y

print("County metadata rows after observed merge:", len(county_meta))
county_meta[["GEOID", "county", "state", "observed_temp_2025_c", "rep_lon", "rep_lat"]].head()

## 5. Load CMIP6 GFDL-ESM4 `tas` projections from the Pangeo CMIP6 catalog

We use:
- `source_id = GFDL-ESM4`
- `variable_id = tas` = near-surface air temperature
- `table_id = Amon` = monthly atmosphere data
- `experiment_id = ssp126, ssp245, ssp585`
- `member_id = r1i1p1f1`

Then we convert monthly temperatures to annual means.

In [ ]:
CATALOG_URL = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
cat = intake.open_esm_datastore(CATALOG_URL)

cat.df.head()

### Helper functions for CMIP6 loading and county sampling

County-level CMIP6 values are estimated by assigning each county representative point to the nearest CMIP6 grid cell. This is appropriate for exploratory mapping, but not a precise county-scale forecast because CMIP6 grids are coarse.

In [ ]:
def _get_coord_name(ds, possible_names):
    for name in possible_names:
        if name in ds.coords or name in ds.dims:
            return name
    raise ValueError(f"Could not find coordinate among {possible_names}. Available coords: {list(ds.coords)}")


def load_cmip6_annual_tas_c(scenario):
    '''
    Load monthly CMIP6 tas for one scenario and convert it to annual mean Celsius.
    '''
    search = cat.search(
        source_id="GFDL-ESM4",
        experiment_id=scenario,
        table_id="Amon",
        variable_id="tas",
        member_id="r1i1p1f1",
    )

    if len(search.df) == 0:
        raise ValueError(f"No CMIP6 catalog entries found for scenario={scenario}")

    # Prefer native or common grid when multiple entries exist.
    # If multiple datasets remain, use the first one.
    print(f"Found {len(search.df)} catalog entries for {scenario}. Using first matching dataset.")

    dsets = search.to_dataset_dict(
        zarr_kwargs={"consolidated": True, "use_cftime": True},
        storage_options={"token": "anon"},
        progressbar=True,
    )

    ds = list(dsets.values())[0]

    lat_name = _get_coord_name(ds, ["lat", "latitude", "y"])
    lon_name = _get_coord_name(ds, ["lon", "longitude", "x"])

    tas = ds["tas"]

    # Subset time first to reduce computation.
    tas = tas.sel(time=slice("2026-01-01", "2036-12-31"))

    # Convert Kelvin to Celsius.
    tas_c = tas - 273.15
    tas_c.name = "projected_temp_c"

    # Annual mean.
    annual = tas_c.resample(time="YS").mean()

    return annual, lat_name, lon_name


def sample_counties_from_cmip6(annual_da, lat_name, lon_name, county_meta, scenario):
    '''
    Sample annual CMIP6 temperature at county representative points using nearest grid cell.
    '''
    # CMIP6 longitudes are often 0–360. Convert county lon if needed.
    lon_values = annual_da[lon_name].values
    lon_min = np.nanmin(lon_values)
    lon_max = np.nanmax(lon_values)

    county_lons = county_meta["rep_lon"].to_numpy()
    if lon_min >= 0 and lon_max > 180:
        county_lons_for_model = (county_lons + 360) % 360
    else:
        county_lons_for_model = county_lons

    county_lats_da = xr.DataArray(
        county_meta["rep_lat"].to_numpy(),
        dims="county_index",
        coords={"county_index": county_meta["GEOID"].to_numpy()},
    )
    county_lons_da = xr.DataArray(
        county_lons_for_model,
        dims="county_index",
        coords={"county_index": county_meta["GEOID"].to_numpy()},
    )

    sampled = annual_da.sel(
        {
            lat_name: county_lats_da,
            lon_name: county_lons_da,
        },
        method="nearest",
    )

    sampled = sampled.assign_coords(
        county_index=county_meta["GEOID"].to_numpy()
    )

    df = sampled.to_dataframe(name="projected_temp_c").reset_index()

    # Extract year from cftime/pandas timestamps.
    df["year"] = df["time"].apply(lambda t: t.year)
    df = df[df["year"].isin(PROJECTED_YEARS)].copy()

    df = df.rename(columns={"county_index": "GEOID"})
    df["scenario"] = scenario
    df["scenario_label"] = SCENARIO_LABELS[scenario]

    keep_cols = ["GEOID", "year", "scenario", "scenario_label", "projected_temp_c"]
    return df[keep_cols]

## 6. Build county-level projection table

This can take several minutes in Colab because CMIP6 data is remote.

In [ ]:
county_projection_parts = []

for scenario in SCENARIOS:
    print("\n==============================")
    print("Processing scenario:", scenario)
    print("==============================")

    annual_da, lat_name, lon_name = load_cmip6_annual_tas_c(scenario)
    sampled_df = sample_counties_from_cmip6(
        annual_da=annual_da,
        lat_name=lat_name,
        lon_name=lon_name,
        county_meta=county_meta,
        scenario=scenario,
    )
    county_projection_parts.append(sampled_df)

county_projection = pd.concat(county_projection_parts, ignore_index=True)

print("County projection rows:", len(county_projection))
county_projection.head()

## 7. Merge CMIP6 projections with observed 2025 county baseline

The main metric is:

`temp_change_from_2025_c = projected_temp_c - observed_temp_2025_c`

In [ ]:
county_attrs = county_meta[
    [
        "GEOID", "county", "state", "state_abbr", "ALAND",
        "observed_temp_2025_f", "observed_temp_2025_c"
    ]
].copy()

county_final_projected = county_projection.merge(
    county_attrs,
    on="GEOID",
    how="left"
)

county_final_projected["temp_change_from_2025_c"] = (
    county_final_projected["projected_temp_c"] -
    county_final_projected["observed_temp_2025_c"]
)

county_final_projected["data_type"] = "projected"

county_final_projected.head()

## 8. Add 2025 observed baseline rows

For line charts, each scenario starts at the same 2025 observed baseline.  
This makes it easy to draw scenario trajectories from the observed starting point.

In [ ]:
baseline_rows = []

for scenario in SCENARIOS:
    temp = county_attrs.copy()
    temp["year"] = 2025
    temp["scenario"] = scenario
    temp["scenario_label"] = SCENARIO_LABELS[scenario]
    temp["projected_temp_c"] = temp["observed_temp_2025_c"]
    temp["temp_change_from_2025_c"] = 0.0
    temp["data_type"] = "observed"
    baseline_rows.append(temp)

county_baseline = pd.concat(baseline_rows, ignore_index=True)

county_final = pd.concat(
    [county_baseline, county_final_projected],
    ignore_index=True
)

county_final = county_final[
    [
        "GEOID", "county", "state", "state_abbr", "ALAND",
        "year", "scenario", "scenario_label", "data_type",
        "observed_temp_2025_f", "observed_temp_2025_c",
        "projected_temp_c", "temp_change_from_2025_c",
    ]
].sort_values(["state", "county", "scenario", "year"])

print("County final rows:", len(county_final))
county_final.head()

## 9. Create state-level table by area-weighting counties

The state map uses state-level summaries. We compute these from county values using county land area (`ALAND`) as weights.

In [ ]:
def weighted_average(group, value_col, weight_col="ALAND"):
    valid = group[value_col].notna() & group[weight_col].notna() & (group[weight_col] > 0)
    if valid.sum() == 0:
        return np.nan
    return np.average(group.loc[valid, value_col], weights=group.loc[valid, weight_col])

state_records = []

group_cols = ["state", "state_abbr", "year", "scenario", "scenario_label", "data_type"]

for keys, group in county_final.groupby(group_cols, dropna=False):
    record = dict(zip(group_cols, keys))
    record["observed_temp_2025_c"] = weighted_average(group, "observed_temp_2025_c")
    record["projected_temp_c"] = weighted_average(group, "projected_temp_c")
    record["temp_change_from_2025_c"] = weighted_average(group, "temp_change_from_2025_c")
    state_records.append(record)

state_final = pd.DataFrame(state_records).sort_values(["state", "scenario", "year"])

print("State final rows:", len(state_final))
state_final.head()

## 10. Create 2036 summary/ranking tables

These are useful for side panels and callouts, e.g. “Top 5 counties under SSP585.”

In [ ]:
county_summary_2036 = county_final[
    (county_final["year"] == 2036) &
    (county_final["data_type"] == "projected")
].copy()

county_summary_2036["rank_within_scenario"] = (
    county_summary_2036
    .groupby("scenario")["temp_change_from_2025_c"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

state_summary_2036 = state_final[
    (state_final["year"] == 2036) &
    (state_final["data_type"] == "projected")
].copy()

state_summary_2036["rank_within_scenario"] = (
    state_summary_2036
    .groupby("scenario")["temp_change_from_2025_c"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

display(state_summary_2036.sort_values(["scenario", "rank_within_scenario"]).head(10))
display(county_summary_2036.sort_values(["scenario", "rank_within_scenario"]).head(10))

## 11. Quality checks

Check missing values and basic ranges before exporting.

In [ ]:
print("County final missing values:")
display(county_final.isna().sum())

print("\nState final missing values:")
display(state_final.isna().sum())

print("\nCounty change summary by scenario/year:")
display(
    county_final
    .groupby(["scenario", "year"])["temp_change_from_2025_c"]
    .describe()
    .round(3)
    .head(20)
)

print("\nState change summary by scenario/year:")
display(
    state_final
    .groupby(["scenario", "year"])["temp_change_from_2025_c"]
    .describe()
    .round(3)
    .head(20)
)

## 12. Export CSV files for D3

In [ ]:
county_csv = OUTPUT_DIR / "county_warming_observed2025_cmip6_2026_2036.csv"
state_csv = OUTPUT_DIR / "state_warming_observed2025_cmip6_2026_2036.csv"
county_summary_csv = OUTPUT_DIR / "county_warming_summary_2036.csv"
state_summary_csv = OUTPUT_DIR / "state_warming_summary_2036.csv"

county_final.to_csv(county_csv, index=False)
state_final.to_csv(state_csv, index=False)
county_summary_2036.to_csv(county_summary_csv, index=False)
state_summary_2036.to_csv(state_summary_csv, index=False)

print("Saved:")
print(county_csv)
print(state_csv)
print(county_summary_csv)
print(state_summary_csv)

## 13. Export GeoJSON files for D3 maps

The data CSVs should be joined in D3 using:
- State map: `state_abbr`
- County map: `GEOID`

For web performance, the 20m Census cartographic boundaries are used.

In [ ]:
states_export = states[["STATEFP", "STUSPS", "NAME", "geometry"]].copy()
states_export = states_export.rename(columns={
    "STUSPS": "state_abbr",
    "NAME": "state"
})

counties_export = county_meta[
    ["GEOID", "STATEFP", "COUNTYFP", "county", "state", "state_abbr", "geometry"]
].copy()

states_geojson = OUTPUT_DIR / "us_states.geojson"
counties_geojson = OUTPUT_DIR / "us_counties.geojson"

states_export.to_file(states_geojson, driver="GeoJSON")
counties_export.to_file(counties_geojson, driver="GeoJSON")

print("Saved:")
print(states_geojson)
print(counties_geojson)

## 14. Zip all outputs for download from Colab

In [ ]:
import shutil

zip_path = shutil.make_archive(
    base_name=str(OUTPUT_DIR),
    format="zip",
    root_dir=str(OUTPUT_DIR)
)

zip_path

## 15. Data interpretation notes for write-up

Use wording like this in your project:

> I use NOAA nClimDiv county average temperature in 2025 as an observed baseline. I then use CMIP6 GFDL-ESM4 near-surface air temperature projections from 2026 to 2036 under SSP126, SSP245, and SSP585. For each county and state, I calculate projected temperature change relative to the observed 2025 baseline.

Important caveat:

> Because the observed baseline and CMIP6 projections come from different data products, the results should be interpreted as an exploratory near-term projection rather than a precise county-level forecast. County-level CMIP6 values are estimated by assigning counties to the nearest coarse model grid cell.

Suggested visualization:
- Initial state-level choropleth map
- Color = `temp_change_from_2025_c`
- Year slider = 2025–2036
- Scenario selector = SSP126 / SSP245 / SSP585
- Click a state → zoom to counties
- Hover county → tooltip
- Click county → line chart across scenarios